# **Introduction to AI Workshop Mini-Project: Text Generation**
In this mini-project, you have to fine-tune a pre-trained GPT-2 language model  **using a text source of your choice**. Complete each task and answer each question below by filling in the blanks (`...`) with one or more lines of code or text.

### **Part 1: Loading the Text Data**

First, we import the libraries we will need.

In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf
import keras
import keras_hub
import re
import matplotlib.pyplot as plt

**Task 01:** Choose and upload a text source to fine-tune the neural network. For possible text sources, see [Project Gutenberg](https://archive.org/details/gutenberg?tab=collection).

In [2]:
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

Saving the-great-gatsby.txt to the-great-gatsby.txt


**Task 02:** Explore the text to get a better understanding of the data.

In [4]:
print("Number of characters:", len(text))
print("\nFirst 500 characters:")
print(text[:500])
print("\n")
print("\nFirst 50 characters:")
print(text[:50])

Number of characters: 272014

First 500 characters:
The Great Gatsby
By F. Scott Fitzgerald

Download free eBooks of classic literature, books and
novels at Planet eBook. Subscribe to our free eBooks blog
and email newsletter.

Then wear the gold hat, if that will move her;
If you can bounce high, bounce for her too,
Till she cry ‘Lover, gold-hatted, high-bouncing lover,
I must have you!’
—THOMAS PARKE D’INVILLIERS



The Great Gatsby

Chapter 1

I

n my younger and more vulnerable years my father gave
me some advice that I’ve been turning ove



First 50 characters:
The Great Gatsby
By F. Scott Fitzgerald

Download 


**Question 01:** How many characters are in the text? What are the first 50 characters?

**Answer:** The .txt file that I uploaded had 272014 characters. There are a lot of extraneous characters as I converted a .pdf file into a .txt file so these are the first 50 characters:

"The Great Gatsby
By F. Scott Fitzgerald

Download "

### **Part 2: Preparing the Text Data**

**Task 03:** Prepare the data by removing any unnecessary whitespace.

In [5]:
text = re.sub(r'\s+', ' ', text).strip()

Next, we divide the text into chunks.

In [6]:
CHUNK_SIZE = 500
OVERLAP = 100
chunks = []
for i in range(0, len(text), CHUNK_SIZE - OVERLAP):
    chunk = text[i:i + CHUNK_SIZE]
    if len(chunk.strip()) > 50:
        chunks.append(chunk)

**Task 04:** Explore the chunks to get a better understanding of the data.

In [7]:
print("Number of text chunks:", len(chunks))
print("\nFirst chunk:")
print(chunks[0])

Number of text chunks: 679

First chunk:
The Great Gatsby By F. Scott Fitzgerald Download free eBooks of classic literature, books and novels at Planet eBook. Subscribe to our free eBooks blog and email newsletter. Then wear the gold hat, if that will move her; If you can bounce high, bounce for her too, Till she cry ‘Lover, gold-hatted, high-bouncing lover, I must have you!’ —THOMAS PARKE D’INVILLIERS  The Great Gatsby Chapter 1 I n my younger and more vulnerable years my father gave me some advice that I’ve been turning over in my m


**Question 02:** How many chunks do we have? What does the first chunk look like?

**Answer:** We have 679 chunks. The first chunk is just some of the copyright information.

### **Part 3: Loading the Pre-Trained GPT-2 Model**

Load the pre-trained GPT-2 model.

In [8]:
SEQUENCE_LENGTH = 128
preprocessor = (
    keras_hub.models.GPT2CausalLMPreprocessor.from_preset(
        "gpt2_base_en",
        sequence_length=SEQUENCE_LENGTH
    )
)
model = keras_hub.models.GPT2CausalLM.from_preset(
    "gpt2_base_en",
    preprocessor=preprocessor
)

### **Part 4: Generating Text with GPT-2**

**Task 05:** Choose a prompt and generate text (100 tokens). Try multiple prompts!

In [9]:
prompt = "The night was young, and the city"
generated_text = model.generate(prompt, max_length=100)
print(generated_text)

The night was young, and the city was full of people, with many of them young and old. The city was a city of young men.

It was the night before, the night before the war. It was the night of war.

The city was the home of the great men of the great army. They were the ones who fought and died to defend it. They would go out and defend it, but they would not do that. The men of the great


**Question 03:** What do you think of the generated text?

**Answer:** The generated text is mostly grammatically correct, but it is very repetitive and doesn't have a clear storyline. For example, the model repeatedly mentions the city, young men, and war without making much progress in the story. Although individual sentences make sense, the overall writing lacks coherence.

### **Part 5: Fine-Tuning GPT-2**

**Task 06:** Fine-tune the model using your text source. Try changing the number of epochs!

In [10]:
BATCH_SIZE = 2

train_dataset = (
    tf.data.Dataset.from_tensor_slices(chunks)
    .shuffle(len(chunks))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    weighted_metrics=["accuracy"]
)

history = model.fit(train_dataset, epochs=1)


340/340 ━━━━━━━━━━━━━━━━━━━━ 137s 197ms/step - accuracy: 0.3600 - loss: 3.4137


### **Part 5: Generating Text After Fine-Tuning**

**Task 07:** Choose a prompt and generate text (100 tokens) with the fine-tuned model. Try multiple prompts!

In [11]:
prompt = "The night was young, and the city"
generated_text = model.generate(prompt, max_length=100)
print(generated_text)


The night was young, and the city was crowded with people of all races and sexes. It was the first night that I went to the West Egg, and the first night that I went to the East Egg. The night was hot and the heat was hot—and the heat was hot because of a little incident that I had with a friend in Chicago that night—but the night was still hot and the heat was hotter than ever. The night was cold—and the heat was cold because


**Question 04:** What do you think of the generated text? Does it match the writing style of your text source?

**Answer:** After fine-tuning GPT-2 on The Great Gatsby, the generated text became more similar to the novel. It started referencing locations such as West Egg and East Egg, which weren't present in the original output. However, the writing was still repetitive and sometimes contradictory, particularly when describing the weather. Overall, fine-tuning helped the model incorporate details from the book, but it still struggled to create a coherent story that fully matched Fitzgerald's writing style.

**Submit your favorite generated text [here](https://forms.gle/FkdbGf92G54sZXx87) (one submission per team).**